### Add widget parameter to notebook.

In [0]:
dbutils.widgets.text("process_date", "2026-02-22")

process_date = dbutils.widgets.get("process_date")

print("Processing date:", process_date)

In [0]:
from pyspark.sql import functions as F

def clean_data(df):
    return (
        df.filter(F.col('price') > 0)
          .dropDuplicates(['user_session', 'event_time'])
          .filter(F.col('category_code').isNotNull())
    )

In [0]:
def add_product_category(df):
    return df.withColumn(
        "product_category",
        F.element_at(
            F.split(F.col("category_code"), "\\."),
            -1
        )
    )

def add_price_tier(df):
    return df.withColumn(
        "price_tier",
        F.when(F.col("price") < 100, "budget")
         .when(F.col("price") < 200, "affordable")
         .when(F.col("price") < 500, "midrange")
         .when(F.col("price") < 1000, "luxury")
         .otherwise("ultra_luxury")
    )

In [0]:
def create_category_features(df):
    return (
        df.filter(F.col("event_type") == "purchase")
          .groupBy("product_category")
          .agg(
              F.count("*").alias("total_items"),
              F.round(F.sum("price"), 2).alias("total_revenue"),
              F.round(F.avg("price"), 2).alias("avg_price"),
              F.min("price").alias("min_price"),
              F.max("price").alias("max_price")
          )
    )


In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/ecommerce/delta/bronze/events")

# Step 1: Clean
silver_clean = clean_data(df)
# Step 2: Enrich
silver_enriched = add_product_category(silver_clean)
silver_enriched = add_price_tier(silver_enriched)
# Step 3: Create Features
features_df = create_category_features(silver_enriched)

features_df.display()

In [0]:
features_df.display()